[![在 Colab 中打开](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Arize-ai/phoenix/blob/main/tutorials/tracing/llama_index_tracing_tutorial.ipynb)


<center>
    <p style="text-align:center">
        <img alt="Phoenix 标志" src="https://storage.googleapis.com/arize-phoenix-assets/assets/phoenix-logo-light.svg" width="200"/>
        <br>
        <a href="https://arize.com/docs/phoenix/">文档</a>
        |
        <a href="https://github.com/Arize-ai/phoenix">GitHub</a>
        |
        <a href="https://join.slack.com/t/arize-ai/shared_invite/zt-3r07iavnk-ammtATWSlF0pSrd1DsMW7g">社区</a>
    </p>
</center>
<h1 align="center">追踪与评估 LlamaIndex 应用</h1>

LlamaIndex 提供高级 API，几行代码即可构建强大的应用，但理解内部执行过程和定位问题往往并不容易。Phoenix 将查询引擎每次调用的底层结构可视化，并根据延迟、token 数及其他评估指标展示存在问题的 `span`，让 LLM 应用具备*可观测性*。

本教程将：
- 使用 LlamaIndex 构建简单查询引擎，通过检索增强生成回答有关 Arize 文档的问题
- 使用全局 `arize_phoenix` 处理器，以 [OpenInference 追踪](https://github.com/Arize-ai/openinference)格式记录数据
- 检查应用的追踪和 span，定位延迟和成本来源
- 将追踪导出为 pandas dataframe，并运行 [LLM 评估](https://arize.com/docs/phoenix/concepts/llm-evals)，衡量查询引擎检索步骤的 precision@k

ℹ️ 本 Notebook 需要 OpenAI API 密钥。

## 1. 安装依赖并导入库

安装 Phoenix、LlamaIndex 和 OpenAI。

In [ ]:
!pip install "arize-phoenix[evals]" "openai>=1" nest-asyncio "openinference-instrumentation-llama-index>=2.0.0" "llama-index-core" "llama-index-llms-openai" "llama-index-embeddings-openai"

导入库。

In [ ]:
import os
from getpass import getpass

import nest_asyncio
import pandas as pd
from llama_index.core import Document, Settings, VectorStoreIndex
from llama_index.embeddings.openai import OpenAIEmbedding
from llama_index.llms.openai import OpenAI
from tqdm import tqdm

nest_asyncio.apply()  # Notebook 环境并发评估所需
pd.set_option("display.max_colwidth", 1000)

## 2. 配置 Phoenix

需要运行 Phoenix 才能收集和可视化 LlamaIndex 应用的追踪数据。有两种方式：

### 方式 1：本地 Phoenix（免费）

在本机安装 Phoenix，并在独立终端启动：

```bash
pip install arize-phoenix
phoenix serve
```

Phoenix 将运行于 `http://localhost:6006`。在浏览器打开此地址即可查看界面。

### 方式 2：Phoenix Cloud（托管）

使用 Phoenix Cloud 托管服务。在 [Phoenix Cloud](https://app.phoenix.arize.com) 注册免费账号并获取 API 密钥，然后设置环境变量：

```bash
export PHOENIX_COLLECTOR_ENDPOINT=your_endpoint_here
export PHOENIX_API_KEY=your_api_key_here
```

埋点代码将自动检测 API 密钥并将追踪发送到 Phoenix Cloud。


## 3. 配置 OpenAI API 密钥

如果尚未通过环境变量设置 OpenAI API 密钥，请在此设置。

In [ ]:
if not (openai_api_key := os.getenv("OPENAI_API_KEY")):
    openai_api_key = getpass("🔑 请输入 OpenAI API 密钥： ")

os.environ["OPENAI_API_KEY"] = openai_api_key

## 4. 构建 LlamaIndex 应用

本示例使用有关 Arize 和 Phoenix 的文档创建简单的 `VectorStoreIndex`。可以将其替换为自己的文档。

下面创建文档集合并构建索引：

In [ ]:
# 配置 LlamaIndex 设置
Settings.llm = OpenAI(model="gpt-4o")
Settings.embed_model = OpenAIEmbedding(model="text-embedding-ada-002")

# 创建关于 Arize 和 Phoenix 的简单演示文档集合
documents = [
    Document(
        text="Phoenix is Arize's open-source observability tool for LLM applications, providing tracing and evaluation capabilities. It helps developers understand LLM application performance and debug issues."
    ),
    Document(
        text="LLM tracing with Phoenix helps you understand the performance and behavior of your language model applications. You can track token usage, latency, and see the complete flow of your RAG pipeline."
    ),
    Document(
        text="Retrieval-augmented generation (RAG) combines information retrieval with language generation for better responses. It allows LLMs to access external knowledge beyond their training data."
    ),
    Document(
        text="Model evaluation is crucial for understanding how well your ML models perform on real-world data. Phoenix provides LLM evals to assess quality, relevance, and hallucinations."
    ),
    Document(
        text="Vector embeddings are numerical representations of text that capture semantic meaning. They enable similarity search and retrieval in RAG systems."
    ),
    Document(
        text="OpenInference is an open standard for LLM application observability. It defines how to capture and store trace data from LLM applications."
    ),
    Document(
        text="Span annotations in Phoenix allow you to add custom metadata and evaluations to your traces, helping you analyze and improve your LLM applications."
    ),
]

通过 `LlamaIndexInstrumentor` 启用 Phoenix 追踪。Phoenix 使用 OpenInference 追踪：这是一种用于捕获和存储 LLM 应用追踪的开源标准，使应用能够无缝接入 Phoenix 等 LLM 可观测性解决方案。

In [ ]:
from openinference.instrumentation.llama_index import LlamaIndexInstrumentor
from phoenix.otel import register

tracer_provider = register(project_name="llamaindex-tracing-tutorial", protocol="http/protobuf")
LlamaIndexInstrumentor().instrument(
    tracer_provider=tracer_provider,
)

现在可以实例化用于检索增强生成（RAG）的查询引擎。查询引擎是 LlamaIndex 中支持对数据提问的通用接口，接收自然语言查询并返回丰富的回答。它构建于检索器之上，也可以组合多个查询引擎实现更高级的能力。

In [ ]:
# 根据文档创建向量索引
print("正在创建向量索引……")
index = VectorStoreIndex.from_documents(documents)

# 创建查询引擎
query_engine = index.as_query_engine()
print("✅ 查询引擎已就绪！")

## 5. 运行查询引擎并在 Phoenix 中查看追踪

创建一些与文档集合相关的示例查询，测试查询引擎：

In [ ]:
# 与文档集合相关的示例查询
queries = [
    "What is Arize and what does it help with?",
    "How does Phoenix help with LLM observability?",
    "What is retrieval-augmented generation?",
    "How can I evaluate my AI Agents?",
    "What are vector embeddings used for?",
    "What is OpenInference?",
    "How do span annotations work in Phoenix?",
    "What kind of monitoring does Arize provide for AI Agents?",
]

print("示例查询：")
for i, query in enumerate(queries, 1):
    print(f"{i}. {query}")
print(f"\n查询总数： {len(queries)}")

运行这些查询，并在 Phoenix 中查看追踪。

In [ ]:
for query in tqdm(queries):
    response = query_engine.query(query)
    print(f"查询： {query}")
    print(f"回复： {response}\n")
    print("-" * 50)

也可以提出自己的问题！

In [ ]:
response = query_engine.query("What is Arize and how can it help me as an AI Engineer?")
print(response)

查询运行时查看 Phoenix 界面，追踪应实时出现。

<img src="https://storage.googleapis.com/arize-phoenix-assets/assets/docs/notebooks/llama-index-knowledge-base-tutorial/Screenshot%202025-09-08%20at%206.53.21%E2%80%AFPM.png" alt="Phoenix 追踪详情视图" style="width:100%; height:auto;">